### Custom statistics for KeypointMoseq Output

In [6]:
import os
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.stats.multitest as smm
import plotly.express as px
import matplotlib.colors as mcolors


def plot_boxplots_with_points(df, global_df, variables, out_dir, alpha=0.05,
                              colorscale_name="Temps"):
    """
    Boxplots + jittered individual points.
    Colors come from a Plotly colormap (Temps, Geyser, etc.).
    """

    # --- build Matplotlib colormap from Plotly ---
    plotly_colors = px.colors.get_colorscale(colorscale_name)

    def _plotly_to_mpl_colors(colorscale):
        """
        Convert Plotly colorscale entries to Matplotlib-friendly RGB tuples or hex.
        colorscale: list like [[pos, 'rgb(r,g,b)'], [pos, '#RRGGBB'], ...]
        """
        mpl_colors = []
        for pos, col in colorscale:
            col = col.strip()
            if col.startswith("rgb"):
                # parse 'rgb(r,g,b)'
                inside = col[col.find("(")+1 : col.find(")")]
                parts = [p.strip() for p in inside.split(",")]
                r, g, b = [int(p) for p in parts]
                mpl_colors.append((r/255.0, g/255.0, b/255.0))
            else:
                # assume already hex or named color
                mpl_colors.append(col)
        return mpl_colors

    mpl_colors_list = _plotly_to_mpl_colors(plotly_colors)

    cmap = mcolors.LinearSegmentedColormap.from_list(
        colorscale_name, mpl_colors_list, N=256
    )

    groups = sorted(df["group"].unique())
    syllables = sorted(df["syllable"].unique())

    # pick N evenly spaced colors from cmap
    group_colors = [cmap(i) for i in np.linspace(0, 1, len(groups))]

    for var in variables:
        fig, ax = plt.subplots(
            figsize=(max(10, len(syllables)*0.6), 6), constrained_layout=True
        )

        positions = np.arange(len(syllables))
        width = 0.15
        offsets = np.linspace(-1.5*width, 1.5*width, len(groups))

        # ---------- boxplots + dots ----------
        for gi, g in enumerate(groups):
            xs = []
            data = []

            for i, syl in enumerate(syllables):
                vals = df.loc[
                    (df["group"] == g) & (df["syllable"] == syl),
                    var
                ].dropna()
                if len(vals) == 0:
                    continue
                data.append(vals.values)
                xs.append(positions[i] + offsets[gi])

            # boxplot
            if data:
                bp = ax.boxplot(
                    data,
                    positions=xs,
                    widths=width,
                    patch_artist=True,
                    showfliers=False,
                )
                for box in bp["boxes"]:
                    box.set_facecolor(group_colors[gi])
                    box.set_alpha(0.5)

            # jittered individual points
            for i, syl in enumerate(syllables):
                vals = df.loc[
                    (df["group"] == g) & (df["syllable"] == syl),
                    var
                ].dropna().values
                if len(vals) == 0:
                    continue

                jitter = np.random.normal(scale=0.02, size=len(vals))
                xpts = positions[i] + offsets[gi] + jitter

                ax.scatter(
                    xpts, vals,
                    color=group_colors[gi],
                    edgecolor="black",
                    s=15,
                    alpha=0.8,
                    linewidth=0.3,
                )

        # ---------- add significance '*' ----------
        sig_syls = set(
            global_df[
                (global_df["variable"] == var) & (global_df["pval"] < alpha)
            ]["syllable"]
        )

        if len(df[var].dropna()) > 0:
            y_min = df[var].min()
            y_max = df[var].max()
            y_range = (y_max - y_min) if y_max > y_min else 1
        else:
            y_min, y_max, y_range = 0, 1, 1

        for syl in sig_syls:
            i = syllables.index(syl)
            y_text = df[df["syllable"] == syl][var].max() + 0.07 * y_range
            ax.text(
                positions[i],
                y_text,
                "*",
                ha="center",
                va="bottom",
                fontsize=18,
                color="black"
            )

        # ---- labels ----
        ax.set_xticks(positions)
        ax.set_xticklabels(syllables)
        ax.set_xlabel("Syllable")
        ax.set_ylabel(var)
        ax.set_title(
            f"{var.capitalize()} by group and syllable\n"
            f"('*' = global effect p < {alpha})"
        )

        # legend
        handles = [
            plt.Line2D([0], [0], color=group_colors[i], lw=6)
            for i in range(len(groups))
        ]
        ax.legend(handles, groups, title="Group", loc="best")

        # save
        fig_path = os.path.join(out_dir, f"boxplot_{var}_with_points.svg")
        fig.savefig(fig_path, dpi=300)
        plt.close(fig)
        print(f"Saved: {fig_path}")
def run_syllable_stats(
    csv_path: str,
    out_dir: str = "stats_results",
    variables=("frequency", "duration"),
    alpha: float = 0.05,
    syll_info_path: str | None = None,
    colorscale_name: str = "Temps",
):
    """
    Run normality tests, ANOVA / Kruskal, post-hoc comparisons and boxplots
    for given variables in stats_df.csv.

    Expected columns in csv:
        - 'group'    : 4 groups (e.g., group1..group4)
        - 'syllable' : integer syllable ID
        - 'frequency', 'duration' : variables to analyze
        - 'name'     : subject / file ID (one row per [name, syllable])

    Optionally merges syllable info from syll_info_path (must contain 'syllable').

    Saves:
        out_dir/global_tests.csv
        out_dir/posthoc_tests.csv
        out_dir/boxplot_<variable>_with_points.svg
    """
    os.makedirs(out_dir, exist_ok=True)

    # ---------------- load main data ----------------
    df = pd.read_csv(csv_path)

    groups = sorted(df["group"].unique())
    syllables = sorted(df["syllable"].unique())

    global_rows = []
    posthoc_rows = []

    # ---------------- how many animals per group ----------------
    print("\n=== Animals per group ===")
    animal_counts = (
        df.groupby("group")["name"]
        .nunique()        # count unique animals in each group
        .sort_index()
    )
    print(animal_counts.to_string())
    print("=========================\n")
    # ---------------- global tests (ANOVA / Kruskal) ----------------
    for var in variables:
        for syl in syllables:
            sub = df[df["syllable"] == syl]

            # --- normality test per group (Shapiro) ---
            normal_by_group = {}
            data_by_group = []
            for g in groups:
                vals = sub.loc[sub["group"] == g, var].dropna().values
                data_by_group.append(vals)

                if len(vals) >= 3:
                    stat_sw, p_sw = stats.shapiro(vals)
                    normal_by_group[g] = bool(p_sw > alpha)
                else:
                    # too few samples to test, treat as non-normal
                    normal_by_group[g] = False

            valid_groups = [g for g, vals in zip(groups, data_by_group) if len(vals) > 0]

            if len(valid_groups) < 2:
                test = "insufficient"
                stat_val = np.nan
                p_val = np.nan
            else:
                all_normal = all(normal_by_group[g] for g in groups)

                if all_normal:
                    # --- one-way ANOVA ---
                    test = "ANOVA"
                    try:
                        stat_val, p_val = stats.f_oneway(
                            *[
                                sub.loc[sub["group"] == g, var]
                                .dropna()
                                .values
                                for g in groups
                            ]
                        )
                    except Exception:
                        test = "ANOVA_error"
                        stat_val, p_val = np.nan, np.nan
                else:
                    # --- Kruskal-Wallis (non-parametric ANOVA equivalent) ---
                    test = "Kruskal"
                    try:
                        stat_val, p_val = stats.kruskal(
                            *[
                                sub.loc[sub["group"] == g, var]
                                .dropna()
                                .values
                                for g in groups
                            ]
                        )
                    except Exception:
                        test = "Kruskal_error"
                        stat_val, p_val = np.nan, np.nan

            row = {
                "variable": var,
                "syllable": syl,
                "test": test,
                "stat": stat_val,
                "pval": p_val,
            }
            for g in groups:
                row[f"normal_{g}"] = normal_by_group[g]

            global_rows.append(row)

    global_df = pd.DataFrame(global_rows)

    # ---------------- post-hoc multiple comparisons ----------------
    for var in variables:
        for syl in syllables:
            g_row = global_df[
                (global_df["variable"] == var) & (global_df["syllable"] == syl)
            ].iloc[0]

            if not np.isfinite(g_row["pval"]) or g_row["pval"] >= alpha:
                continue

            test_type = g_row["test"]
            sub = df[df["syllable"] == syl]
            pair_results = []

            for g1, g2 in combinations(groups, 2):
                x1 = sub.loc[sub["group"] == g1, var].dropna().values
                x2 = sub.loc[sub["group"] == g2, var].dropna().values

                if len(x1) == 0 or len(x2) == 0:
                    continue

                if test_type.startswith("ANOVA"):
                    # Welch's t-test
                    stat_val, p_pair = stats.ttest_ind(
                        x1, x2, equal_var=False, nan_policy="omit"
                    )
                    stat_name = "t"
                else:
                    # Mann-Whitney U for non-parametric case
                    stat_val, p_pair = stats.mannwhitneyu(
                        x1, x2, alternative="two-sided"
                    )
                    stat_name = "U"

                pair_results.append(
                    {
                        "variable": var,
                        "syllable": syl,
                        "global_test": test_type,
                        "group1": g1,
                        "group2": g2,
                        "stat_name": stat_name,
                        "stat": stat_val,
                        "p_uncorrected": p_pair,
                    }
                )

            if not pair_results:
                continue

            # FDR-BH correction within this syllable+variable
            pvals = [r["p_uncorrected"] for r in pair_results]
            reject, p_adj, _, _ = smm.multipletests(
                pvals, alpha=alpha, method="fdr_bh"
            )

            for r, p_corr, rej in zip(pair_results, p_adj, reject):
                r["p_adj"] = p_corr
                r["significant"] = bool(rej)
                posthoc_rows.append(r)

    posthoc_df = pd.DataFrame(posthoc_rows)

    # ---------------- merge syll_info if provided ----------------
    if syll_info_path is not None and os.path.exists(syll_info_path):
        syll_info = pd.read_csv(syll_info_path)
        if "syllable" in syll_info.columns:
            # merge into global and posthoc tables
            global_df = global_df.merge(syll_info, on="syllable", how="left")
            posthoc_df = posthoc_df.merge(syll_info, on="syllable", how="left")
            print(f"Merged syll_info from: {syll_info_path}")
        else:
            print("Warning: syll_info.csv does not contain a 'syllable' column. Skipping merge.")

    # ---------------- save tables ----------------
    global_path = os.path.join(out_dir, "global_tests.csv")
    posthoc_path = os.path.join(out_dir, "posthoc_tests.csv")
    global_df.to_csv(global_path, index=False)
    posthoc_df.to_csv(posthoc_path, index=False)
    print(f"Saved global tests to:   {global_path}")
    print(f"Saved post-hoc tests to: {posthoc_path}")

    # ---------------- plots (boxplots + points) ----------------
    plot_boxplots_with_points(
        df=df,
        global_df=global_df,
        variables=variables,
        out_dir=out_dir,
        alpha=alpha,
        colorscale_name=colorscale_name,
    )

    return global_df, posthoc_df


if __name__ == "__main__":
    # Adjust these paths as needed
    csv_path = "/Users/annateruel/Desktop/wanhui/stats_df.csv"
    out_dir = "/Users/annateruel/Desktop/wanhui/stats_results"
    syll_info_path = "/Users/annateruel/Desktop/wanhui/syll_info.csv"  # <-- update or None

    global_df, posthoc_df = run_syllable_stats(
        csv_path,
        out_dir=out_dir,
        variables=("frequency", "duration"),
        alpha=0.05,
        syll_info_path=syll_info_path,
        colorscale_name="Temps",   # or "Geyser"
    )


=== Animals per group ===
group
group1    26
group2    75
group3    24
group4    23

Saved global tests to:   /Users/annateruel/Desktop/wanhui/stats_results/global_tests.csv
Saved post-hoc tests to: /Users/annateruel/Desktop/wanhui/stats_results/posthoc_tests.csv
Saved: /Users/annateruel/Desktop/wanhui/stats_results/boxplot_frequency_with_points.svg
Saved: /Users/annateruel/Desktop/wanhui/stats_results/boxplot_duration_with_points.svg


Now we're going to run sepparate statistics for each individual ROI, to assess which are the most frequent syllables on each selected ROI. 

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

def load_rect_rois_from_h5(roi_h5_path):
    """
    Load rectangle ROIs saved by napari to an .h5 file and return
    a DataFrame with one row per ROI:
        roi_index, x_min, x_max, y_min, y_max
    """
    shapes = pd.read_hdf(roi_h5_path)

    if not {"index", "axis-0", "axis-1"}.issubset(shapes.columns):
        raise ValueError(f"Unexpected ROI columns: {shapes.columns}")

    rois = []
    for idx, grp in shapes.groupby("index"):
        xs = grp["axis-0"].values  # x = axis-1
        ys = grp["axis-1"].values  # y = axis-0
        rois.append(
            {
                "roi_index": int(idx),
                "x_min": xs.min(),
                "x_max": xs.max(),
                "y_min": ys.min(),
                "y_max": ys.max(),
            }
        )

    return pd.DataFrame(rois)
def build_roi_dict(roi_dir, suffix="_roi.h5"):
    """
    Find all ROI files in roi_dir matching *<suffix> (e.g. *_roi.h5)
    and return a dict: { base_name : roi_df }.

    base_name is the filename with `suffix` stripped, e.g.
      '12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2'
    """
    roi_dict = {}
    pattern = os.path.join(roi_dir, f"*{suffix}")
    for path in glob.glob(pattern):
        base = os.path.basename(path)
        base_name = base[:-len(suffix)]  # strip "_roi.h5"
        roi_dict[base_name] = load_rect_rois_from_h5(path)
    return roi_dict
def extract_base_name(fullname: str) -> str:
    """
    Remove DLC / snapshot suffix from a MoSeq recording name.

    Example:
      '12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2DLC_DekrW32_3chamDec3shuffle1_snapshot_200_filtered'
        -> '12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2'
    """
    markers = [
        "DLC_DekrW32",
        "DLC",
        "sDLC",
        "snapshot",
    ]
    base = fullname
    for m in markers:
        if m in base:
            base = base.split(m)[0]
    return base.rstrip("_- ")
def assign_roi_index_all_by_basename(moseq_df, roi_dict):
    """
    Add an `roi_index` column to moseq_df, using a different ROI set
    for each recording.

    Matching is done by a 'base name' extracted from moseq_df['name']
    that should match the ROI filename prefix.
    """
    moseq_df = moseq_df.copy()
    moseq_df["roi_index"] = -1

    # compute base_name for all frames
    moseq_df["base_name"] = moseq_df["name"].apply(extract_base_name)

    unique_basenames = moseq_df["base_name"].unique()

    for roi_base, rois in roi_dict.items():
        # find matching basename(s)
        candidates = [b for b in unique_basenames if roi_base in b or b in roi_base]

        if len(candidates) == 0:
            print(f"Warning: ROIs for '{roi_base}' but no matching recording in moseq_df")
            continue

        if len(candidates) > 1:
            print(f"Multiple matches for ROI '{roi_base}': {candidates}; using first.")
        chosen_base = candidates[0]

        mask_rec = moseq_df["base_name"] == chosen_base
        n_frames = mask_rec.sum()
        if n_frames == 0:
            print(f"Warning: matched basename '{chosen_base}' but no frames in moseq_df")
            continue

        print(f"Assigning ROI '{roi_base}' → recording '{chosen_base}', frames: {n_frames}")

        # IMPORTANT: use correct column names: centroid_x / centroid_y
        x = moseq_df.loc[mask_rec, "centroid_x"].values
        y = moseq_df.loc[mask_rec, "centroid_y"].values

        roi_idx_sub = np.full(x.shape[0], -1, dtype=int)

        for _, r in rois.iterrows():
            m = (
                (x >= r["x_min"]) & (x <= r["x_max"]) &
                (y >= r["y_min"]) & (y <= r["y_max"])
            )
            roi_idx_sub[m] = int(r["roi_index"])

        moseq_df.loc[mask_rec, "roi_index"] = roi_idx_sub

    return moseq_df



In [7]:
import os
import glob
import pandas as pd
import numpy as np

def get_napari_roi_cols(df):
    """
    For ROIDrawer .h5 files:

        axis-0 = x (horizontal)
        axis-1 = y (vertical)
    """
    cols = df.columns

    idx_col   = next(
        c for c in cols
        if "index" in str(c).lower() and "vertex" not in str(c).lower()
    )
    axis0_col = next(c for c in cols if "axis-0" in str(c))  # x
    axis1_col = next(c for c in cols if "axis-1" in str(c))  # y

    x_col = axis0_col   # x
    y_col = axis1_col   # y
    return idx_col, x_col, y_col
def load_rect_rois_from_h5(roi_h5_path):
    shapes = pd.read_hdf(roi_h5_path)
    idx_col, x_col, y_col = get_napari_roi_cols(shapes)

    rois = []
    for idx, grp in shapes.groupby(idx_col):
        xs = grp[x_col].values   # x
        ys = grp[y_col].values   # y
        rois.append(
            {
                "roi_index": int(idx),
                "x_min": xs.min(),
                "x_max": xs.max(),
                "y_min": ys.min(),
                "y_max": ys.max(),
            }
        )
    return pd.DataFrame(rois)
def build_roi_dict(roi_dir, suffix="_roi.h5"):
    """
    Find all ROI files in roi_dir matching *<suffix> (e.g. *_roi.h5)
    and return a dict: { base_name : roi_df }.

    base_name is the filename with `suffix` stripped, e.g.
      '12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2'
    """
    roi_dict = {}
    pattern = os.path.join(roi_dir, f"*{suffix}")
    for path in glob.glob(pattern):
        base = os.path.basename(path)
        base_name = base[:-len(suffix)]  # strip "_roi.h5"
        roi_dict[base_name] = load_rect_rois_from_h5(path)
    return roi_dict
def extract_base_name(fullname: str) -> str:
    """
    Remove DLC / snapshot suffix from a MoSeq recording name.

    Example:
      '12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2DLC_DekrW32_3chamDec3shuffle1_snapshot_200_filtered'
        -> '12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2'
    """
    markers = ["DLC_DekrW32", "DLC", "sDLC", "snapshot"]
    base = fullname
    for m in markers:
        if m in base:
            base = base.split(m)[0]
    return base.rstrip("_- ")
def assign_roi_index_all_by_basename(moseq_df, roi_dict):
    """
    Add an `roi_index` column to moseq_df, using a different ROI set
    for each recording.

    Matching is done by a 'base name' extracted from moseq_df['name']
    that should match the ROI filename prefix.
    """
    moseq_df = moseq_df.copy()
    moseq_df["roi_index"] = -1

    # compute base_name for all frames
    moseq_df["base_name"] = moseq_df["name"].apply(extract_base_name)

    unique_basenames = moseq_df["base_name"].unique()

    for roi_base, rois in roi_dict.items():
        # find matching basename(s)
        candidates = [b for b in unique_basenames if b == roi_base]

        if len(candidates) == 0:
            print(f"Warning: ROIs for '{roi_base}' but no matching recording in moseq_df")
            continue

        if len(candidates) > 1:
            print(f"Multiple matches for ROI '{roi_base}': {candidates}; using first.")
        chosen_base = candidates[0]

        mask_rec = moseq_df["base_name"] == chosen_base
        n_frames = int(mask_rec.sum())
        if n_frames == 0:
            print(f"Warning: matched basename '{chosen_base}' but no frames in moseq_df")
            continue

        print(f"Assigning ROI '{roi_base}' → recording '{chosen_base}', frames: {n_frames}")

        # IMPORTANT: centroid_x / centroid_y are in the same image coords
        x = moseq_df.loc[mask_rec, "centroid_x"].values
        y = moseq_df.loc[mask_rec, "centroid_y"].values

        roi_idx_sub = np.full(x.shape[0], -1, dtype=int)

        for _, r in rois.iterrows():
            m = (
                (x >= r["x_min"]) & (x <= r["x_max"]) &
                (y >= r["y_min"]) & (y <= r["y_max"])
            )
            roi_idx_sub[m] = int(r["roi_index"])

        moseq_df.loc[mask_rec, "roi_index"] = roi_idx_sub

    return moseq_df



In [ ]:
import keypoint_moseq as kpms
from keypoint_moseq.analysis import compute_moseq_df

project_dir = '/Users/annateruel/Desktop/wanhui/'
model_name  = '2025_07_10-18_30_37'

moseq_df = compute_moseq_df(
    project_dir,
    model_name,
    fps=30,
    smooth_heading=True,
)

roi_dir = "/Users/annateruel/Desktop/wanhui/dlc"
roi_dict = build_roi_dict(roi_dir, suffix="_roi.h5")

moseq_df = assign_roi_index_all_by_basename(moseq_df, roi_dict)

print("\n=== ROI DISTRIBUTION PER RECORDING ===")

for base in moseq_df["base_name"].unique():
    sub = moseq_df[moseq_df["base_name"] == base]
    print(f"\nRecording: {base}")
    print("Total frames:", len(sub))
    print(sub["roi_index"].value_counts(dropna=False))

In [ ]:
base = "12-11-23_Exp7SalineCtrl_d211101-2-cliped_v2"

sub = moseq_df[moseq_df["base_name"] == base]
print("Frames in this recording:", len(sub))
print(sub["roi_index"].value_counts())
print(sub["roi_index"].value_counts(normalize=True))  # percentages

In [8]:
import keypoint_moseq as kpms
from keypoint_moseq.analysis import compute_moseq_df

project_dir = '/Users/annateruel/Desktop/wanhui/'
model_name  = '2025_07_10-18_30_37'

moseq_df = compute_moseq_df(
    project_dir,
    model_name,
    fps=30,
    smooth_heading=True,
)

roi_dir = "/Users/annateruel/Desktop/wanhui/rois"
roi_dict = build_roi_dict(roi_dir, suffix="_roi.h5")
moseq_df  = assign_roi_index_all_by_basename(moseq_df, roi_dict)
print(moseq_df["roi_index"].value_counts().head())

Assigning ROI '3802Test_12-cliped' → recording '3802Test_12-cliped', frames: 8968
Assigning ROI '3305Test_10-cliped' → recording '3305Test_10-cliped', frames: 8971
Assigning ROI '9304Test-6-5minCliped_2' → recording '9304Test-6-5minCliped_2', frames: 8973
Assigning ROI '8901Test-2-cliped_2' → recording '8901Test-2-cliped_2', frames: 8968
Assigning ROI 'exp14-15102-2_v2' → recording 'exp14-15102-2_v2', frames: 9001
Assigning ROI 'exp15-13303-5_v2' → recording 'exp15-13303-5_v2', frames: 9005
Assigning ROI 'exp15-m1102-2_v2' → recording 'exp15-m1102-2_v2', frames: 1473
Assigning ROI 'exp14-m1502-4_v2' → recording 'exp14-m1502-4_v2', frames: 9005
Assigning ROI '5803Test_8-cliped' → recording '5803Test_8-cliped', frames: 8969
Assigning ROI 'exp15-14202-4_v2' → recording 'exp15-14202-4_v2', frames: 9005
Assigning ROI 'exp15-m1101-8' → recording 'exp15-m1101-8', frames: 9001
Assigning ROI 'exp7-11904-4_v2' → recording 'exp7-11904-4_v2', frames: 9000
Assigning ROI '9203Test-13-cliped_2' → rec

Plotting a video to double check

In [ ]:
import cv2
import os

WANHUI_DIR = "/Users/annateruel/Desktop/wanhui/"
VIDEO_DIR  = os.path.join(WANHUI_DIR, "dlc")
ROI_DIR    = "/Users/annateruel/Desktop/wanhui/rois"
OUT_DIR    = os.path.join(WANHUI_DIR, "debug_videos")
os.makedirs(OUT_DIR, exist_ok=True)

def make_roi_highlight_video(
    rec_base_name,
    moseq_df,
    rect_roi_df,
    video_dir=VIDEO_DIR,
    out_dir=OUT_DIR,
    ext=".mp4",
    max_frames=None,
):
    """
    Create a video for one recording where:
      - centroid is shown as a yellow dot
      - all ROIs are drawn
      - the ROI that the animal is in on that frame is highlighted
        with a specific color; others are grey.
    """

    # subset dataframe for this recording
    sub = moseq_df[moseq_df["base_name"] == rec_base_name].copy()
    if sub.empty:
        print(f"[SKIP] No frames in moseq_df for '{rec_base_name}'")
        return

    sub = sub.sort_values("frame_index")
    frame_to_row = sub.set_index("frame_index")

    # video paths
    video_path = os.path.join(video_dir, rec_base_name + ext)
    if not os.path.exists(video_path):
        print(f"[SKIP] Video not found: {video_path}")
        return

    out_path = os.path.join(out_dir, rec_base_name + "_roi_debug.mp4")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[SKIP] Could not open video: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"[DEBUG] {rec_base_name} frame size: {width}x{height}")
    print("[DEBUG] ROI bounds from rect_roi_df:")
    print(rect_roi_df.describe())
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    # colors for ROIs (BGR)
    roi_colors = [
        (0,   0, 255),   # red
        (0, 255,   0),   # green
        (255, 0,   0),   # blue
        (0, 255, 255),   # yellow
        (255, 0, 255),   # magenta
        (255, 255, 0),   # cyan
    ]
    grey = (180, 180, 180)

    n_frames_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_idx_df = int(sub["frame_index"].max())
    n_frames = min(n_frames_video, max_idx_df + 1)
    if max_frames is not None:
        n_frames = min(n_frames, max_frames)

    print(f"[VIDEO] {rec_base_name}: writing {n_frames} frames → {out_path}")

    for frame_idx in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx not in frame_to_row.index:
            writer.write(frame)
            continue

        row = frame_to_row.loc[frame_idx]
        x = float(row["centroid_x"])
        y = float(row["centroid_y"])
        roi_idx = int(row.get("roi_index", -1))

        # draw all ROIs
        for _, r in rect_roi_df.iterrows():
            rid = int(r["roi_index"])
            x1, x2 = int(r["x_min"]), int(r["x_max"])
            y1, y2 = int(r["y_min"]), int(r["y_max"])

            if rid == roi_idx and roi_idx >= 0:
                color = roi_colors[rid % len(roi_colors)]
                thick = 4
            else:
                color = grey
                thick = 2

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thick)

        # draw centroid
        if not np.isnan(x) and not np.isnan(y):
            cv2.circle(frame, (int(x), int(y)), 4, (0, 255, 255), -1)

        # text with ROI index (can be -1)
        cv2.putText(
            frame,
            f"ROI: {roi_idx}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

        writer.write(frame)

    cap.release()
    writer.release()
    print("  Saved:", out_path)
def export_all_roi_debug_videos(moseq_df, roi_dir=ROI_DIR,
                                video_dir=VIDEO_DIR, out_dir=OUT_DIR,
                                ext=".mp4", max_frames=None):
    """
    Build ROI dict from roi_dir and export a debug video
    for every recording that has both:
      - entries in moseq_df
      - a ROI file in roi_dir
    """

    # ensure base_name column
    if "base_name" not in moseq_df.columns:
        moseq_df = moseq_df.copy()
        moseq_df["base_name"] = moseq_df["name"].apply(extract_base_name)

    roi_dict = build_roi_dict(roi_dir)

    # loop over each base_name that has an ROI definition
    for base_name, rect_roi_df in roi_dict.items():
        if base_name not in moseq_df["base_name"].values:
            print(f"[SKIP] '{base_name}' has ROIs but no entries in moseq_df")
            continue

        make_roi_highlight_video(
            base_name,
            moseq_df=moseq_df,
            rect_roi_df=rect_roi_df,
            video_dir=video_dir,
            out_dir=out_dir,
            ext=ext,
            max_frames=max_frames,
        )

export_all_roi_debug_videos(moseq_df, max_frames=500)

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt


def load_bodypart_xy(dlc_path, bodypart="nose"):
    """
    Load x,y for one bodypart from a DLC file (.h5 or .csv).
    Handles both MultiIndex columns and flattened columns.
    """
    ext = os.path.splitext(dlc_path)[1].lower()
    if ext == ".h5":
        df = pd.read_hdf(dlc_path)
    else:  # assume csv
        df = pd.read_csv(dlc_path)

    # --- DLC with MultiIndex columns (scorer, bodypart, coord) ---
    if isinstance(df.columns, pd.MultiIndex):
        # usually: level 0 = scorer, 1 = bodypart, 2 = coord
        scorers = df.columns.get_level_values(0).unique()
        scorer = scorers[0]

        try:
            x = df[scorer, bodypart, "x"].to_numpy()
            y = df[scorer, bodypart, "y"].to_numpy()
        except KeyError as e:
            raise KeyError(f"Bodypart '{bodypart}' not found in {dlc_path}") from e

    # --- Flattened columns, e.g. nose_x, nose_y, nose_likelihood ---
    else:
        x_col = None
        y_col = None
        for col in df.columns:
            cl = str(col).lower()
            if bodypart.lower() in cl and "x" == cl.split("_")[-1]:
                x_col = col
            if bodypart.lower() in cl and "y" == cl.split("_")[-1]:
                y_col = col

        if x_col is None or y_col is None:
            raise KeyError(f"Could not find x/y columns for '{bodypart}' in {dlc_path}")

        x = df[x_col].to_numpy()
        y = df[y_col].to_numpy()

    return x, y


def plot_tracking_for_directory(
    dlc_dir,
    pattern="*.h5",
    bodypart="nose",
    save_dir=None,
    invert_y=True,
):
    """
    Plot XY trajectories for `bodypart` for all DLC files in `dlc_dir`.

    One figure per file.
    """
    dlc_paths = sorted(glob.glob(os.path.join(dlc_dir, pattern)))
    if not dlc_paths:
        print("No DLC files found in:", dlc_dir)
        return

    if save_dir is None:
        save_dir = dlc_dir
    os.makedirs(save_dir, exist_ok=True)

    for path in dlc_paths:
        print("Processing:", os.path.basename(path))
        x, y = load_bodypart_xy(path, bodypart=bodypart)

        plt.figure(figsize=(5, 5))
        plt.plot(x, y, linewidth=0.5)
        if invert_y:
            plt.gca().invert_yaxis()  # camera coordinates: (0,0) at top-left

        plt.xlabel("x (pixels)")
        plt.ylabel("y (pixels)")
        plt.title(f"{os.path.basename(path)} – {bodypart} trajectory")
        plt.tight_layout()

        out_name = os.path.splitext(os.path.basename(path))[0] + f"_{bodypart}_traj.png"
        out_path = os.path.join(save_dir, out_name)
        plt.savefig(out_path, dpi=200)
        plt.close()
        print("  Saved:", out_path)


# ------- use it like this --------
dlc_dir = "/path/to/your/dlc/files"  # e.g. "/Users/annateruel/Desktop/wanhui/dlc"
plot_tracking_for_directory(
    dlc_dir,
    pattern="*filtered.h5",  # or "*.csv" or "*DLC*.h5"
    bodypart="nose",
    save_dir=os.path.join(dlc_dir, "tracks"),
)

In [13]:
from keypoint_moseq.io import load_results
from jax_moseq.utils import get_durations, get_frequencies
def compute_stats_df_per_roi(
    project_dir,
    model_name,
    moseq_df,
    min_frequency=0.005,
    fps=30,
    num_factors=1,
):
    """
    Summary stats for syllable frequencies and kinematic values,
    separately for each ROI (uses moseq_df['roi_index']).

    Returns
    -------
    stats_df : DataFrame
        one row per (name, roi_index, group1..groupN, syllable)
        with columns:
            frequency, duration, occupancy, persistence,
            heading_*, angular_velocity_*, velocity_px_s_*
    """
    # ----- choose grouping keys: name + roi + group1..groupN -----
    base_groupby = ["name", "roi_index"]
    factor_cols = [f"group{i}" for i in range(1, num_factors + 1)]
    groupby = base_groupby + factor_cols

    # ---- global filter: drop very rare syllables (same as before) ----
    results_dict = load_results(project_dir, model_name)
    syllables_all = {k: res["syllable"] for k, res in results_dict.items()}
    freqs_global = get_frequencies(syllables_all)
    syll_include = np.where(freqs_global > min_frequency)[0]

    # keep only those syllables
    filtered_df = moseq_df[moseq_df["syllable"].isin(syll_include)].copy()

    # ---- kinematic features per ROI+group+syllable ----
    features = (
        filtered_df
        .groupby(groupby + ["syllable"])[["heading", "angular_velocity", "velocity_px_s"]]
        .agg(["mean", "std", "min", "max"])
    )
    features.columns = ["_".join(col).strip() for col in features.columns.values]
    features.reset_index(inplace=True)

    # ---- syllable "episodes" & durations / persistence per ROI ----
    # episodes are defined by onset True then consecutive frames with same syllable
    trials = filtered_df["onset"].cumsum()

    # duration (mean frames per episode)
    durations = (
        filtered_df
        .groupby(groupby + ["syllable", trials])["onset"].count()
        .groupby(groupby + ["syllable"]).mean()
    )
    durations.name = "duration"
    durations = durations.fillna(0).reset_index()[groupby + ["syllable", "duration"]]

    # persistence (median frames per episode)
    persistence = (
        filtered_df
        .groupby(groupby + ["syllable", trials])["onset"].count()
        .groupby(groupby + ["syllable"]).median()
    )
    persistence.name = "persistence"
    persistence = persistence.fillna(0).reset_index()[groupby + ["syllable", "persistence"]]

    # ---- occupancy: fraction of frames in ROI spent in each syllable ----
    occupancies = filtered_df.groupby(groupby + ["syllable"]).size()
    total_frames_per_group = filtered_df.groupby(groupby)["syllable"].size()
    occupancies = occupancies / total_frames_per_group
    occupancies.name = "occupancy"
    occupancies = occupancies.reset_index()[groupby + ["syllable", "occupancy"]]

    # ---- frequency: fraction of syllable *episodes* per ROI ----
    episodes = (
        filtered_df[filtered_df["onset"]]
        .groupby(groupby + ["syllable"])["onset"].size()
    )
    episodes.name = "n_events"
    # normalize by total events per group+ROI
    freq = episodes.groupby(level=list(range(len(groupby)))).transform("sum")
    frequency = (episodes / freq).reset_index()
    frequency = frequency.rename(columns={"onset": "n_events"})
    frequency = frequency.rename(columns={0: "frequency"})  # in case name lost
    frequency["frequency"] = episodes / freq
    frequency = frequency.reset_index()
    # columns after reset_index: level_0.., fix:
    cols = groupby + ["syllable", "frequency"]
    frequency = frequency[cols]

    # ---- merge all stats ----
    stats_df = features
    stats_df = stats_df.merge(frequency, on=groupby + ["syllable"], how="left")
    stats_df = stats_df.merge(durations, on=groupby + ["syllable"], how="left")
    stats_df = stats_df.merge(occupancies, on=groupby + ["syllable"], how="left")
    stats_df = stats_df.merge(persistence, on=groupby + ["syllable"], how="left")

    return stats_df

In [16]:
stats_df = compute_stats_df_per_roi(
    project_dir,
    model_name,
    moseq_df,
    min_frequency=0.005,
    fps=30,
    num_factors=1,
)
# stats_df = stats_df.rename(columns={"group": "group"})
stats_df.to_csv("/Users/annateruel/Desktop/wanhui/stats_df_with_roi.csv", index=False)

KeyError: 'group1'

In [ ]:
from pathlib import Path

base_out = Path("/Users/annateruel/Desktop/wanhui/stats_results_roi")

for roi in sorted(stats_df["roi_index"].unique()):
    sub = stats_df[stats_df["roi_index"] == roi].copy()
    csv_roi = base_out.parent / f"stats_df_roi{roi}.csv"
    out_dir_roi = base_out / f"roi{roi}"
    out_dir_roi.mkdir(parents=True, exist_ok=True)

    sub.to_csv(csv_roi, index=False)

    # reuse your run_syllable_stats function that expects a CSV path
    run_syllable_stats(
        csv_path=str(csv_roi),
        out_dir=str(out_dir_roi),
        variables=("frequency", "duration"),
        alpha=0.05,
    )